## Benchmark Run Script (Reference)

```bash
cd /Users/RyanLandvater/Programming_Projects/FHIR_Testing
DYLD_LIBRARY_PATH=local/lib ./build/bench/bench/bench_harness --warmup-iterations 2 --iterations 10 --db "host=localhost port=5432 dbname=benchmark user=bench password=bench"
```

Optional explicit runs (same as current default):

```bash
cd /Users/RyanLandvater/Programming_Projects/FHIR_Testing
DYLD_LIBRARY_PATH=local/lib ./build/bench/bench/bench_harness --warmup-iterations 2 --iterations 10 --db "host=localhost port=5432 dbname=benchmark user=bench password=bench" --runs 10
```

```bash
PGPASSWORD=bench /opt/homebrew/Cellar/libpq/17.4/bin/psql -h localhost -p 5432 -U bench -d benchmark -c "TRUNCATE benchmark_results, benchmark_runs RESTART IDENTITY CASCADE;" && echo "Done"
```


Deprecated old command path removed. Use the benchmark DB command shown above.

# FastFHIR vs JSON-FHIR Benchmark Results

This notebook loads and analyzes benchmark results from the C++ harness.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import os
from sqlalchemy import create_engine, text

# Configuration - read from environment or use benchmark defaults
db_host = os.getenv('POSTGRES_HOST', 'localhost')
db_port = os.getenv('POSTGRES_PORT', '5432')
db_name = os.getenv('POSTGRES_DB', 'benchmark')
db_user = os.getenv('POSTGRES_USER', 'bench')
db_pass = os.getenv('POSTGRES_PASSWORD', 'bench')

db_url = f"postgresql+psycopg2://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}"
print(f"Connecting to {db_user}@{db_host}:{db_port}/{db_name}")

## Load Results from Database

In [ ]:
try:
    engine = create_engine(db_url, pool_pre_ping=True)
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Connected successfully (SQLAlchemy)")
except Exception as e:
    print(f"Connection failed: {e}")
    engine = None

## Summary Statistics

In [ ]:
if 'engine' in locals() and engine is not None:
    query = """
    SELECT 
        br.id,
        run_id,
        arm,
        stage,
        duration_us,
        target_mb,
        patients_in_bundle,
        br.created_at
    FROM benchmark_results br
    ORDER BY run_id DESC, target_mb, arm, stage
    LIMIT 10000
    """
    
    # Store full result set for analysis (no automatic printing).
    df = pd.read_sql(query, engine)
    
    # Convenience DataFrames for manual inspection when needed.
    df_preview = df.head(20).copy()
    latest_run_id = int(df['run_id'].max()) if len(df) > 0 else None
    latest_run_df = df[df['run_id'] == latest_run_id].copy() if latest_run_id is not None else pd.DataFrame()
else:
    df = pd.DataFrame()
    df_preview = pd.DataFrame()
    latest_run_df = pd.DataFrame()
    latest_run_id = None

## Summary Statistics

In [ ]:
if 'df' in locals() and len(df) > 0:
    summary = df.groupby(['arm', 'stage'])['duration_us'].agg(['mean', 'std', 'min', 'max']).round(2)
    arm_totals = df.groupby('arm')['duration_us'].agg(['sum', 'mean', 'count']).round(2)
    
    # Keep optional preview frames ready for manual review.
    summary_preview = summary.reset_index().copy()
    arm_totals_preview = arm_totals.reset_index().copy()
else:
    summary = pd.DataFrame()
    arm_totals = pd.DataFrame()
    summary_preview = pd.DataFrame()
    arm_totals_preview = pd.DataFrame()
summary_preview

## Performance Comparison - Serialization

In [ ]:
if 'df' in locals() and len(df) > 0:
    use_latest_run_only = True
    source_df = latest_run_df if use_latest_run_only and 'latest_run_df' in locals() and len(latest_run_df) > 0 else df
    stage1_data = source_df[source_df['stage'] == 'stage1_serialize'].copy()

    if len(stage1_data) > 0:
        show_smoke_arms = False
        if not show_smoke_arms:
            stage1_data = stage1_data[stage1_data['arm'].isin(['fastfhir', 'json_fhir'])].copy()

        arms = sorted(stage1_data['arm'].unique())
        target_mbs = sorted(stage1_data['target_mb'].unique())
        n_arms = len(arms)
        n_targets = len(target_mbs)

        colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
        arm_colors = {arm: colors[i] for i, arm in enumerate(arms)}

        width = 0.35
        fig, ax = plt.subplots(figsize=(14, 6))

        for ai, arm in enumerate(arms):
            offset = (ai - (n_arms - 1) / 2) * width
            positions = [i + 1 + offset for i in range(n_targets)]
            violin_data = [
                stage1_data[(stage1_data['arm'] == arm) & (stage1_data['target_mb'] == mb)]['duration_us'].values
                for mb in target_mbs
            ]
            valid = [(pos, d) for pos, d in zip(positions, violin_data) if len(d) >= 2]
            if valid:
                vpos, vdata = zip(*valid)
                parts = ax.violinplot(list(vdata), positions=list(vpos), widths=width * 0.9,
                                      showmedians=True, showextrema=True)
                color = arm_colors[arm]
                for pc in parts['bodies']:
                    pc.set_facecolor(color)
                    pc.set_alpha(0.7)
                for key in ('cmedians', 'cmaxes', 'cmins', 'cbars'):
                    parts[key].set_color(color)
                    parts[key].set_linewidth(1.5)
            ax.fill_between([], [], color=arm_colors[arm], alpha=0.7, label=arm)

        ax.set_xticks(range(1, n_targets + 1))
        ax.set_xticklabels([f'{mb} MB' for mb in target_mbs])
        ax.set_xlabel('Bundle Size (MB)', fontsize=12)
        ax.set_ylabel('Serialization Time (microseconds)', fontsize=12)
        ax.set_title('Serialization Performance Distribution by Arm', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_yscale('log')
        plt.tight_layout()
        plt.show()
    else:
        print('No serialization data found')

## Performance Comparison - Query

In [ ]:
if 'df' in locals() and len(df) > 0:
    use_latest_run_only = True
    source_df = latest_run_df if use_latest_run_only and 'latest_run_df' in locals() and len(latest_run_df) > 0 else df
    stage3_data = source_df[source_df['stage'] == 'stage3_query'].copy()

    if len(stage3_data) > 0:
        show_smoke_arms = False
        if not show_smoke_arms:
            stage3_data = stage3_data[stage3_data['arm'].isin(['fastfhir', 'json_fhir'])].copy()

        arms = sorted(stage3_data['arm'].unique())
        target_mbs = sorted(stage3_data['target_mb'].unique())
        n_arms = len(arms)
        n_targets = len(target_mbs)

        colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
        arm_colors = {arm: colors[i] for i, arm in enumerate(arms)}

        width = 0.35
        fig, ax = plt.subplots(figsize=(14, 6))

        for ai, arm in enumerate(arms):
            offset = (ai - (n_arms - 1) / 2) * width
            positions = [i + 1 + offset for i in range(n_targets)]
            violin_data = [
                stage3_data[(stage3_data['arm'] == arm) & (stage3_data['target_mb'] == mb)]['duration_us'].values
                for mb in target_mbs
            ]
            valid = [(pos, d) for pos, d in zip(positions, violin_data) if len(d) >= 2]
            if valid:
                vpos, vdata = zip(*valid)
                parts = ax.violinplot(list(vdata), positions=list(vpos), widths=width * 0.9,
                                      showmedians=True, showextrema=True)
                color = arm_colors[arm]
                for pc in parts['bodies']:
                    pc.set_facecolor(color)
                    pc.set_alpha(0.7)
                for key in ('cmedians', 'cmaxes', 'cmins', 'cbars'):
                    parts[key].set_color(color)
                    parts[key].set_linewidth(1.5)
            ax.fill_between([], [], color=arm_colors[arm], alpha=0.7, label=arm)

        ax.set_xticks(range(1, n_targets + 1))
        ax.set_xticklabels([f'{mb} MB' for mb in target_mbs])
        ax.set_xlabel('Bundle Size (MB)', fontsize=12)
        ax.set_ylabel('Query Time (microseconds)', fontsize=12)
        ax.set_title('Query Performance Distribution by Arm', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_yscale('log')
        plt.tight_layout()
        plt.show()
    else:
        print('No query data found')

## Cleanup

In [ ]:
if 'engine' in locals() and engine is not None:
    engine.dispose()
    print("Database engine disposed")